# Problem 1 (Fast convolution) 2.5 pts
* 1D
Implement fast multiplication of a Toeplitz matrix by a given vector with $\mathcal{O}(n \log n)$ complexity. Make sure that you do not form the Toeplitz matrix itself!
Now you are able to implement Problem 1 from the Problem Set 1 without truncating the signal. Convolve your signal with the filter $T$ using your fast multiplication algorithm (set $\alpha=\frac{1}{20}$). Play the signal.
## Bonus
Find convolution of the Lena $n\times n$ image and the following filter $$
T_{i_1j_1i_2j_2} \equiv T_{i_1-j_1,i_2-j_2} = \frac{\alpha}{\pi} e^{-\alpha \left[(i_1 - j_1)^2 + (i_2 - j_2)^2 \right]}, \quad i_1,j_1, i_2, j_2 = 1,\dots, n, \quad \alpha = \frac{1}{50}
$$ with $\mathcal{O}(n^2 \log n^2)$ complexity. Plot the resulting image (`plt.imshow` might be helpful for this task). What is the complexity of naive summation?

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def toeplitz_mul(t_row, v):
    result = None
    # IMPLEMENT ME: here
    
    return result

In [ ]:
# reading
rate, audio = wav.read('data/E=MC2.wav')

# plotting
plt.plot(audio)
plt.ylabel('Amplitude')
plt.xlabel('Time')
plt.title('E=MC^2')
plt.show()

# playing
Audio(audio, rate=rate)

In [ ]:
alpha = 1 / 20.0
t_vec = np.sqrt(alpha / np.pi) * np.exp(-alpha * np.arange(len(s1)) ** 2)
t_vec = np.hstack((t_vec, t_vec[1:len(t_vec)][::-1]))

pl.figure(figsize=(10,10))
pl.plot(s1[0:100], label='signal')
pl.title('Convolution')

convolution = toeplitz_mul(t_vec, s1)

pl.plot(convolution[0:100], label='conv with alpha = {0}'.format(alpha))
    
pl.legend(loc='upper right')
pl.ylabel('Voltage')
pl.xlabel('N')
pl.show()

In [ ]:
from scipy import signal
from scipy import misc

def toeplitz2d_mul(f, i):
    # IMPLEMENT ME: here
    result = None
    
    return result


lena = lena()
alpha = 1 / 50.0

n = lena.shape[0]

In [ ]:
T = None # FIX ME: define T properly

conv = toeplitz2d_mul(T, lena)
fig, ax = plt.subplots(2, figsize=(8, 8))
ax[0].imshow(lena, cmap='gray')
ax[0].set_title('Original Lena')
ax[1].imshow(conv[0:n, 0:n], cmap='gray')
ax[1].set_title('Convolved Lena')

# Problem 2 2.5 pts

* Implement Jacobi, Gauss-Seidel, SOR (play with different $\omega$) iterations
* Compare the methods with each other - plot residuals vs number of iterations for N = 1000

In [ ]:
def jacobi(A, f, x0, eps=1e-6, maxiter=1000):
    x = x0.copy()
    dx = np.zeros_like(f)
    it = 0
    n = len(x)
    while it < maxiter:
        # IMPLEMENT ME: here
    raise RuntimeError('Maximum number of iterations exceeded')

In [ ]:
n = 10
A = np.random.rand(n, n) # random matrix
A = A + np.diag(A.sum(axis=1)) # make sure that diagonal dominates
x = np.ones(n) # exact solution
f = A.dot(x)
x0 = np.zeros_like(x)
xres, it = jacobi(A, f, x0)
print('Done in %d iterations' % it)
print('||x - x_res|| =', np.linalg.norm(xres - x))

In [ ]:
def sor(A, f, x0, omega, eps=1e-6, maxiter=1000):
    x = x0.copy()
    n = len(x)
    it = 0
    while it < maxiter:
        # IMPLEMENT ME: here
    raise RuntimeError('Maximum number of iterations exceeded') 

# Problem 3 1 pt

* Implement CG and Gradient Descent (play with different $\alpha$)$
* Compare the methods with each other - plot residuals vs number of iterations for `N = 1000`

# Problem 4 1 pt
Consider following matrices:

## Case 1:
`A_indefinite = np.array([[1, 2], [2, 1]])`

## Case 2:
`A_nonsym_dd = np.array([[4, 1], [0, 3]])`

## Case 3:
`A_bad = np.array([[1, 2], [3, 1]])`

Find their eigenvalues $\lambda$, and run each of the solvers above for a small random right-hand side (RHS), plot residuals vs number of iterations for `N = 1000`. Explain the results. Could you have predicted them before running the methods?

## Linearizing the Laplacian Operator

This section walks through how the continuous operator `∇²` becomes a
discrete matrix `A` acting on a flattened vector, so that the PDE
`-k∇²T = q` becomes a linear system `Ax = f`.

---

### 1. Start: The Continuous Operator

$$
\nabla^2 T = \frac{\partial^2 T}{\partial x^2} + \frac{\partial^2 T}{\partial y^2}
$$

This acts on a continuous function `T(x,y)` — it cannot be represented as a
finite matrix until we restrict `T` to a finite set of points.

---

### 2. Discretize: Finite-Difference Stencil

On a uniform grid with spacing `h`, the second derivative is approximated
(via Taylor expansion, see earlier derivation) as:

$$
\frac{\partial^2 T}{\partial x^2}\Big|_{i,j} \approx
\frac{T_{i+1,j} - 2T_{i,j} + T_{i-1,j}}{h^2}
$$

$$
\frac{\partial^2 T}{\partial y^2}\Big|_{i,j} \approx
\frac{T_{i,j+1} - 2T_{i,j} + T_{i,j-1}}{h^2}
$$

Adding these gives the **5-point stencil**:

$$
\nabla^2 T_{i,j} \approx
\frac{T_{i+1,j} + T_{i-1,j} + T_{i,j+1} + T_{i,j-1} - 4T_{i,j}}{h^2}
$$

This is still a *2D array operation* - one equation per grid node, but
expressed with 2D indices `(i,j)`, not yet a matrix-vector product.

---

### 3. Flatten: 2D Indices → 1D Vector

To write this as `Ax`, every grid point `T[i,j]` must map to a single
entry `x[k]` in a 1D vector. Use **row-major (natural) ordering**:

$$
k = i \cdot N + j
$$

Under this mapping, neighbors translate as:

| 2D neighbor | 1D index |
|---|---|
| `T[i, j]`     | `x[k]`     |
| `T[i+1, j]`   | `x[k + N]` |
| `T[i-1, j]`   | `x[k - N]` |
| `T[i, j+1]`   | `x[k + 1]` |
| `T[i, j-1]`   | `x[k - 1]` |

---

### 4. Write One Row of the Matrix

Substituting into `-k∇²T = q`, the stencil equation at node `k` becomes:

$$
\frac{4k}{h^2}\,x_k \;-\; \frac{k}{h^2}\,x_{k+1} \;-\; \frac{k}{h^2}\,x_{k-1}
\;-\; \frac{k}{h^2}\,x_{k+N} \;-\; \frac{k}{h^2}\,x_{k-N} \;=\; q_k
$$

This is **row `k`** of the linear system `Ax = f`:

- Diagonal entry `A[k, k] = 4k/h²`
- Off-diagonal entries `A[k, k±1] = A[k, k±N] = -k/h²`
- All other entries in row `k` are `0`

Repeating this for every interior node builds the full sparse matrix `A`.

---

### 5. Elegant Shortcut: Kronecker Sum

Instead of filling `A` row by row, the 2D Laplacian can be built directly
from two copies of the **1D Laplacian**, using a Kronecker sum:

**1D Laplacian** (`N × N`, tridiagonal):

$$
L_{1D} = \frac{1}{h^2}\,\text{tridiag}(1, -2, 1)
$$

**2D Laplacian** (`N² × N²`, via Kronecker sum):

$$
L_{2D} = L_{1D} \otimes I \;+\; I \otimes L_{1D}
$$

where `I` is the `N × N` identity matrix and `⊗` is the Kronecker product.
This single expression reproduces the *exact same* matrix as the row-by-row
derivation above — the Kronecker structure automatically applies the 1D
stencil along rows, then along columns, and sums the two contributions.

```python
import numpy as np
from scipy.sparse import diags, kron, identity

def laplacian_2d(N, h):
    main_diag = -2.0 * np.ones(N)
    off_diag  =  1.0 * np.ones(N - 1)
    L_1D = diags([off_diag, main_diag, off_diag], [-1, 0, 1]) / h**2

    I = identity(N)
    L_2D = kron(L_1D, I) + kron(I, L_1D)
    return L_2D
```

# Problem 5 3 pts

## Steady-State Thermal Analysis of a CPU Die

## Overview

You will model steady-state heat conduction in a 2D CPU die, discretize the
governing PDE on a uniform grid, and solve the resulting linear system using
five iterative methods: **Jacobi**, **Gauss-Seidel**, **SOR**,
**Gradient Descent**, and **Conjugate Gradient**.

---

## 1. Physical Setup

A silicon die is modeled as a 2D square domain:

- Domain: `Ω = [0, L] × [0, L]`, with `L = 0.02 m` (20 mm)
- Thermal conductivity: `k = 150 W / (m·K)`

Three functional blocks generate heat within the die:

| Block            | Region (x-range, y-range)     | Power density `q` (W/m³) |
|------------------|--------------------------------|---------------------------|
| Core A           | `[4, 8] mm × [4, 8] mm`         | `5 × 10⁷`                 |
| Core B           | `[12, 16] mm × [4, 8] mm`       | `5 × 10⁷`                 |
| Memory Controller| `[6, 14] mm × [14, 18] mm`      | `2 × 10⁷`                 |

Everywhere else in the domain, `q(x, y) = 0`.

---

## 2. Governing Equation

Steady-state heat conduction with internal heat generation is governed by the
**Poisson equation**:

$$
-k \nabla^2 T(x, y) = q(x, y), \qquad (x, y) \in \Omega
$$

### Boundary Condition

The die edges are attached to a heatsink held at ambient temperature
(**Dirichlet boundary condition**):

$$
T(x, y) = T_{amb} = 25\,°C, \qquad (x, y) \in \partial\Omega
$$

## 3. Task

1. Discretize the PDE above on a uniform `N × N` grid, $N \in [5, 10, 100]$.
2. Solve the resulting linear system `Ax = b` using:
   - Jacobi
   - Gauss-Seidel
   - SOR (sweep several values of `ω`)
   - Gradient Descent (sweep several values of `α`)
   - Conjugate Gradient
3. Report:
   - **Peak die temperature** and its `(x, y)` location
   - Whether any point exceeds the throttle threshold `T_throttle = 95 °C`
   - Which method converges fastest, in both **iteration count** and
     **wall-clock time**
4. Find maximal $N$ you could solve on your PC for each method

---

## 4. Constants Reference

| Symbol        | Meaning                          | Value              |
|---------------|------------------------------------|---------------------|
| `L`           | Die side length                    | `0.02 m`            |
| `k`           | Thermal conductivity               | `150 W/(m·K)`       |
| `T_amb`       | Ambient / heatsink temperature     | `25 °C`             |
| `T_throttle`  | Thermal throttle threshold         | `95 °C`             |
| `N`           | Grid resolution (nodes per side)   | variable (e.g. 200, 1000) |
| `h`           | Grid spacing                       | `L / (N - 1)`       |
